In [1]:
import sys
import subprocess
from pathlib import Path

REPO = Path("ProPainter")

# Clonar ProPainter.
if not REPO.exists():
    subprocess.check_call([
        "git",
        "clone",
        "https://github.com/sczhou/ProPainter.git",
        str(REPO),
    ])

# Actualizar pip en el mismo Python del notebook.
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "pip",
    "setuptools",
    "wheel",
])

# PyTorch con CUDA 12.6 para la RTX 4090.
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "torch==2.6.0",
    "torchvision==0.21.0",
    "--index-url",
    "https://download.pytorch.org/whl/cu126",
])

# Dependencias de ProPainter.
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-r",
    str(REPO / "requirements.txt"),
])

# Confirmar que PyTorch ve la GPU.
import torch

print("PyTorch:", torch.__version__)
print("CUDA de PyTorch:", torch.version.cuda)
print("CUDA disponible:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "PyTorch no detectó la GPU. Reinicia el kernel y ejecuta nuevamente."
    )

print("GPU:", torch.cuda.get_device_name(0))
print(
    "VRAM:",
    round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3,
        2,
    ),
    "GB",
)

PyTorch: 2.6.0+cu126
CUDA de PyTorch: 12.6
CUDA disponible: True
GPU: NVIDIA GeForce RTX 4090
VRAM: 23.99 GB


In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

NOMBRE_VIDEO = "video30min-11to22"

REPO = Path("ProPainter").resolve()

CARPETA_VIDEO = Path("Videos") / NOMBRE_VIDEO
FRAMES = CARPETA_VIDEO / "frames"
MASKS = CARPETA_VIDEO / "masks"
NO_HUD = CARPETA_VIDEO / "no_hud"
TEMPORAL = CARPETA_VIDEO / "_propainter_temporal"

# Limpiar ejecuciones anteriores.
if TEMPORAL.exists():
    shutil.rmtree(TEMPORAL)

if NO_HUD.exists():
    shutil.rmtree(NO_HUD)

TEMPORAL.mkdir(parents=True)
NO_HUD.mkdir(parents=True)

# Configuración CUDA.
entorno = os.environ.copy()
entorno["CUDA_VISIBLE_DEVICES"] = "0"
entorno["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Configuración conservadora pero con buena calidad para RTX 4090 de 24 GB.
comando = [
    sys.executable,
    "inference_propainter.py",

    "--video", str(FRAMES.resolve()),
    "--mask", str(MASKS.resolve()),
    "--output", str(TEMPORAL.resolve()),

    "--width", "1280",
    "--height", "720",

    "--fp16",
    "--save_frames",

    "--neighbor_length", "10",
    "--ref_stride", "10",
    "--subvideo_length", "40",

    "--mask_dilation", "3",
    "--raft_iter", "20",
]

subprocess.run(
    comando,
    cwd=str(REPO),
    env=entorno,
    check=True,
)

# Encontrar la carpeta de frames generada por ProPainter.
carpetas_con_png = [
    carpeta
    for carpeta in [TEMPORAL, *TEMPORAL.rglob("*")]
    if carpeta.is_dir() and list(carpeta.glob("*.png"))
]

if not carpetas_con_png:
    raise RuntimeError("No se encontraron frames generados por ProPainter.")

# Elegir la carpeta que produjo más imágenes.
carpeta_resultado = max(
    carpetas_con_png,
    key=lambda carpeta: len(list(carpeta.glob("*.png"))),
)

resultados = sorted(carpeta_resultado.glob("*.png"))
nombres_originales = sorted(FRAMES.glob("*.png"))

if len(resultados) != len(nombres_originales):
    raise RuntimeError(
        f"Cantidad diferente: {len(resultados)} resultados "
        f"para {len(nombres_originales)} frames."
    )

# Guardar con exactamente los mismos nombres de los frames originales.
for resultado, frame_original in zip(resultados, nombres_originales):
    shutil.copy2(
        resultado,
        NO_HUD / frame_original.name,
    )

shutil.rmtree(TEMPORAL)

print("ProPainter terminó correctamente.")
print("Frames procesados:", len(resultados))
print("Salida:", NO_HUD.resolve())